# 42 · Trục thời gian — bốn phép đo **chặn**, trước khi dựng bất cứ thứ gì

**CPU thuần, chỉ đọc.** Không đụng GPU ⇒ chạy song song với `38c` được.

Mỗi cell trả lời một câu **có/không** đã được đặt ngưỡng trước, dùng đúng số học đã lập:

> Đòn bẩy chạm `k` nhãn phải nhấc mỗi nhãn `0.0007 × 40 / k` chỉ để vượt biên độ seed.
> `k` là thứ quyết định, không phải độ lớn trên từng nhãn.

| cell | câu hỏi | ngưỡng đặt trước |
|---|---|---|
| `C1` | bao nhiêu study có **phim chụp trước**? | `≥ 60%` ⇒ `k ≥ 24`, đi tiếp · `< 40%` ⇒ đóng |
| `C2` | `lab_*_7d` có thật sự khác `lab_*_24h`? | `≥ 90%` trùng ⇒ cột 7d là **bản sao** |
| `C3` | bao nhiêu study đủ dữ liệu để **tính được xu hướng**? | `< 30%` ⇒ xu hướng không định nghĩa được cho đa số |
| `C4` | MIMIC còn giữ **mùa vụ** không? | nhãn nhiễm trùng dao động theo tháng mà nhãn đối chứng thì không |

`C4` là phép thử quyết định: so một nhãn **đáng lẽ theo mùa** với một nhãn **đáng lẽ không**. Cả hai
phẳng ⇒ MIMIC đã xoá mùa vụ khi dịch ngày, và hướng đó đóng luôn.

## C0 · Nạp — chỉ đọc, không ghi gì

In [1]:
import os, json, re
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

P_BASE = os.path.join(OUT_DIR, 'cxrlt_base.parquet')
P_IDX  = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
P_EHR  = os.path.join(OUT_DIR, 'cxrlt_ehr.parquet')
P_EMB  = os.path.join(EMB_DIR, 'emb_index.parquet')
for p in (P_BASE, P_IDX, P_EHR, P_EMB):
    assert os.path.exists(p), f'khong thay {p}'

BASE  = pd.read_parquet(P_BASE)
BASE['index_time'] = pd.to_datetime(BASE.index_time)
INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']
TIERN = {t: set(INV['tier'][t]) for t in ('head', 'medium', 'tail')}
IDX   = pd.read_parquet(P_IDX, columns=['dicom_id', 'study_id', 'split'] + LAB40)
ORDER = pd.read_parquet(P_EMB, columns=['dicom_id', 'study_id'])

# study CO anh dung duoc — phim truoc phai la phim ta THAT SU doc duoc, khong phai chi co ID
HAVE_IMG = set(ORDER.study_id.unique())
B = BASE[BASE.study_id.isin(HAVE_IMG)].copy()
print(f'{len(BASE):,} study trong base | {len(B):,} co anh dung duoc | '
      f'{B.subject_id.nunique():,} benh nhan')
print(f'setting: {B.setting.value_counts().to_dict()}')
print(f'split:   {B.split.value_counts().to_dict()}')
NOISE_MACRO = 0.0007      # bien do seed cua macro, tu REPORT_STAGE4_NB35 §0.2
need = lambda k: NOISE_MACRO * 40 / max(k, 1e-9)

Mounted at /content/drive
227,835 study trong base | 227,835 co anh dung duoc | 65,379 benh nhan
setting: {'inpatient': 110892, 'ed': 77801, 'outpatient': 39142}
split:   {'train': 155823, 'test': 48212, 'val': 23800}


## C1 · Phim chụp trước — phép đo **chặn** cho hướng longitudinal

`k` hiệu dụng = `40 × tỉ lệ study có phim trước`. Nếu tỉ lệ thấp thì ngưỡng lift/nhãn tăng vọt và
hướng này đắt hơn mọi thứ khác mà không hơn về hình dạng.

Tách theo **split** vì nếu `val` có ít phim trước hơn `train` thì lợi ích không chuyển sang được.

In [2]:
B = B.sort_values(['subject_id', 'index_time']).reset_index(drop=True)
g = B.groupby('subject_id', sort=False)
B['n_truoc']   = g.cumcount()
B['t_truoc']   = g.index_time.shift(1)
B['gap_ngay']  = (B.index_time - B.t_truoc).dt.total_seconds() / 86400.0
B['co_truoc']  = B.n_truoc > 0

frac = B.co_truoc.mean()
print(f'=== PHIM CHUP TRUOC (cung benh nhan, TRUOC index_time, CO anh dung duoc) ===')
print(f'  study co it nhat 1 phim truoc : {B.co_truoc.sum():,} / {len(B):,}  ({frac:.1%})')
print(f'  so phim truoc — trung vi {B.n_truoc.median():.0f} | TB {B.n_truoc.mean():.1f} '
      f'| toi da {B.n_truoc.max():,}')
_gp = B.gap_ngay.dropna()
print(f'  khoang cach (ngay) — p25 {_gp.quantile(.25):.1f} | trung vi {_gp.median():.1f} '
      f'| p75 {_gp.quantile(.75):.1f} | p90 {_gp.quantile(.90):.0f}')

print('\n  theo SPLIT (neu val thap hon train thi loi ich khong chuyen sang duoc):')
print(B.groupby('split').co_truoc.agg(['mean', 'size']).rename(
      columns={'mean': 'ty_le_co_truoc', 'size': 'n_study'}).to_string(
      float_format=lambda v: f'{v:.1%}'))
print('\n  theo SETTING:')
print(B.groupby('setting').co_truoc.agg(['mean', 'size']).rename(
      columns={'mean': 'ty_le_co_truoc', 'size': 'n_study'}).to_string(
      float_format=lambda v: f'{v:.1%}'))

# gan day thi huu ich hon: mot phim cach 3 nam noi rat it ve "cai nay co MOI khong"
for d in (30, 90, 365):
    f2 = (B.gap_ngay <= d).mean()
    print(f'  co phim truoc trong {d:>3} ngay: {f2:.1%}   -> k = {40*f2:.0f}, '
          f'can {need(40*f2):.4f}/nhan')

k_eff = 40 * frac
print(f'\n  k hieu dung = 40 x {frac:.1%} = {k_eff:.0f}')
print(f'  lift moi nhan can de macro dich MOT bien do seed: {need(k_eff):.4f}')
print('  ---')
print(f'  >= 60% -> DI TIEP (k >= 24, nguong <= 0.0012/nhan)')
print(f'  <  40% -> DONG   (k < 16, nguong >= 0.0018/nhan, dat gap boi xu huong lab)')
print(f'  KET LUAN: {"DI TIEP" if frac >= 0.60 else "DONG" if frac < 0.40 else "VUNG XAM"}')

=== PHIM CHUP TRUOC (cung benh nhan, TRUOC index_time, CO anh dung duoc) ===
  study co it nhat 1 phim truoc : 162,456 / 227,835  (71.3%)
  so phim truoc — trung vi 2 | TB 5.7 | toi da 157
  khoang cach (ngay) — p25 0.9 | trung vi 3.3 | p75 56.8 | p90 305

  theo SPLIT (neu val thap hon train thi loi ich khong chuyen sang duoc):
       ty_le_co_truoc  n_study
split                         
test            72.9%    48212
train           70.8%   155823
val             71.3%    23800

  theo SETTING:
            ty_le_co_truoc  n_study
setting                            
ed                   51.3%    77801
inpatient            87.1%   110892
outpatient           66.5%    39142
  co phim truoc trong  30 ngay: 49.5%   -> k = 20, can 0.0014/nhan
  co phim truoc trong  90 ngay: 56.3%   -> k = 23, can 0.0012/nhan
  co phim truoc trong 365 ngay: 65.4%   -> k = 26, can 0.0011/nhan

  k hieu dung = 40 x 71.3% = 29
  lift moi nhan can de macro dich MOT bien do seed: 0.0010
  ---
  >= 60% -> DI TIE

## C2 · `lab_*_7d` có phải bản sao của `lab_*_24h` không

Cả hai lấy **giá trị gần nhất** trong cửa sổ của mình, mà `24h ⊂ 7d`. Nên nếu có xét nghiệm trong
24h thì giá trị gần nhất của **cả hai** là cùng một lần đo ⇒ hai cột **bằng nhau từng con số**.

Nếu đúng vậy thì trong 90 cột "giá trị lâm sàng" mà `nb35` đo là đóng góp 0, một phần đáng kể
là **cột trùng lặp** — chứ không phải thông tin bị mô hình bỏ qua.

In [3]:
E = pd.read_parquet(P_EHR)
V24 = sorted(c for c in E.columns if re.fullmatch(r'lab_.+_24h', c))
print(f'{len(V24)} cot gia tri lab_*_24h\n')
rows = []
for c24 in V24:
    c7 = c24[:-3] + '7d'
    if c7 not in E.columns: continue
    a, b = E[c24], E[c7]
    m = a.notna()
    if not m.any(): continue
    rows.append(dict(lab=c24[4:-4], n_co_24h=int(m.sum()),
                     trung=float(np.isclose(a[m], b[m], rtol=0, atol=1e-9).mean()),
                     n_chi_co_7d=int((a.isna() & b.notna()).sum())))
T = pd.DataFrame(rows).sort_values('n_co_24h', ascending=False)
print(T.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
_tb = float(np.average(T.trung, weights=T.n_co_24h))
_extra = int(T.n_chi_co_7d.sum())
print(f'\n  ty le TRUNG (co trong so theo so mau): {_tb:.1%}')
print(f'  so o ma cot 7d them thong tin that (24h rong, 7d co): {_extra:,}')
print('  ---')
print(f'  >= 90% trung -> cot lab_*_7d gan nhu la BAN SAO. Xu huong KHONG duoc ma hoa o dau ca.')
print(f'  KET LUAN: {"BAN SAO" if _tb >= 0.90 else "co thong tin rieng"}')

18 cot gia tri lab_*_24h

        lab  n_co_24h  trung  n_chi_co_7d
 hemoglobin    137348  1.000        17216
        rbc    136590  1.000        17536
        wbc    136122  1.000        17745
   platelet    136035  1.000        17662
 creatinine    123847  1.000        18935
        bun    123546  1.000        18764
    glucose    122514  1.000        16657
     sodium    122060  1.000        18054
   chloride    122049  1.000        18025
  potassium    121964  1.000        18069
bicarbonate    121177  1.000        18267
        inr     74902  1.000        36812
    lactate     56263  1.000        36837
  bilirubin     42290  1.000        38670
    albumin     27737  1.000        41366
   troponin      9241  1.000        14235
   ntprobnp      4217  1.000        11595
     ddimer      1429  1.000         1944

  ty le TRUNG (co trong so theo so mau): 100.0%
  so o ma cot 7d them thong tin that (24h rong, 7d co): 378,389
  ---
  >= 90% trung -> cot lab_*_7d gan nhu la BAN SAO. Xu huo

## C3 · Xu hướng có **tính được** không

Không cần truy vấn lại BigQuery: cột `n_lab_*_{24h,7d}` đã đếm sẵn **số lần đo**. Muốn có độ dốc
thì phải có `≥ 2` lần đo trong cửa sổ. Đây là **chặn trên** của mọi đặc trưng xu hướng.

In [4]:
N24 = sorted(c for c in E.columns if re.fullmatch(r'n_lab_.+_24h', c))
N7  = sorted(c for c in E.columns if re.fullmatch(r'n_lab_.+_7d',  c))
print(f'{len(N24)} cot dem 24h | {len(N7)} cot dem 7d\n')
rows = []
for c in N24:
    c7 = c[:-3] + '7d'
    v, v7 = E[c].fillna(0), (E[c7].fillna(0) if c7 in E.columns else pd.Series(0, index=E.index))
    rows.append(dict(lab=c[6:-4], co_1_trong_24h=float((v >= 1).mean()),
                     co_2_trong_24h=float((v >= 2).mean()), co_2_trong_7d=float((v7 >= 2).mean())))
T3 = pd.DataFrame(rows).sort_values('co_1_trong_24h', ascending=False)
print(T3.to_string(index=False, float_format=lambda v: f'{v:.1%}'))
_p24, _p7 = float(T3.co_2_trong_24h.max()), float(T3.co_2_trong_7d.max())
print(f'\n  xet nghiem THUAN LOI NHAT: >=2 lan do trong 24h {_p24:.1%} | trong 7d {_p7:.1%}')
print('  ---')
print('  Do doc chi DINH NGHIA DUOC cho phan study nay. Phan con lai phai dien mot gia tri')
print('  "khong co xu huong" — tuc lai them mot cot MAT, dung loai thong tin `nb35` do la')
print('  dang mang toan bo lift. Khong hai, nhung cung khong phai thong tin moi.')
print(f'  KET LUAN: {"xu huong khong dinh nghia duoc cho da so" if _p7 < 0.30 else "dinh nghia duoc cho da so"}')

18 cot dem 24h | 18 cot dem 7d

        lab  co_1_trong_24h  co_2_trong_24h  co_2_trong_7d
 hemoglobin           60.3%           14.9%          45.5%
        rbc           60.0%           13.0%          44.7%
        wbc           59.7%           13.0%          44.6%
   platelet           59.7%           13.4%          44.7%
 creatinine           54.4%           16.5%          44.2%
        bun           54.2%           16.3%          44.0%
    glucose           53.8%           18.1%          43.8%
     sodium           53.6%           17.4%          43.6%
   chloride           53.6%           17.4%          43.6%
  potassium           53.5%           17.4%          43.6%
bicarbonate           53.2%           15.8%          43.2%
        inr           32.9%            7.0%          31.7%
    lactate           24.7%            8.3%          21.9%
  bilirubin           18.6%            2.4%          18.3%
    albumin           12.2%            0.9%          10.7%
   troponin            4

## C4 · MIMIC còn giữ **mùa vụ** không — phép thử quyết định

MIMIC-IV dịch ngày về tương lai theo từng bệnh nhân. Câu hỏi là phép dịch đó có **giữ tháng trong
năm** hay không — và ta trả lời được **bằng dữ liệu**, không cần đọc tài liệu:

```
so mot nhan DANG LE theo mua      (viem phoi, nhiem trung)
voi mot nhan DANG LE khong        (tim to, thiet bi ho tro, xuong)
```

Cả hai phẳng ⇒ mùa vụ đã bị xoá, hướng đóng. Nhãn nhiễm trùng dao động mà đối chứng thì không
⇒ mùa vụ còn nguyên và **dùng được ngay** (một cột `tháng`, gần như miễn phí).

In [5]:
S = (IDX.merge(B[['study_id', 'index_time']], on='study_id', how='inner'))
S['thang'] = S.index_time.dt.month
print(f'{len(S):,} anh co ca nhan lan index_time')
print(f'phan bo thang: {S.thang.value_counts().sort_index().to_dict()}')

MUA_RE = r'pneumon|tubercul|infect|consolidat|abscess|empyema|edema'
DOI_RE = r'cardiomegal|support device|fracture|scoliosis|calcif|hernia|nodule'
G_MUA = [l for l in LAB40 if re.search(MUA_RE, l, re.I)]
G_DOI = [l for l in LAB40 if re.search(DOI_RE, l, re.I)]
print('')
print(f'nhom DANG LE theo mua ({len(G_MUA)}): {G_MUA}')
print(f'nhom DOI CHUNG        ({len(G_DOI)}): {G_DOI}')
assert G_MUA and G_DOI, f'mot nhom rong -> sua regex cho khop {LAB40}'

_th = S.thang.to_numpy()
_Y  = {l: S[l].to_numpy('float64') for l in G_MUA + G_DOI}

def _stat(th):
    '''trung vi cua ty le dong/he, nhom theo mua TRU nhom doi chung.
       Ty le tu chuan hoa theo do pho bien nen so sanh duoc giua cac nhan.'''
    d, h = np.isin(th, [12, 1, 2]), np.isin(th, [6, 7, 8])
    r = lambda l: (_Y[l][d].mean() / _Y[l][h].mean()) if _Y[l][h].mean() > 0 else np.nan
    return (np.nanmedian([r(l) for l in G_MUA]) - np.nanmedian([r(l) for l in G_DOI]))

obs = _stat(_th)
# Nguong KHONG duoc dat bang tay: sinh phan phoi NULL bang cach TRAO thang giua cac anh.
# Trao thang pha moi lien he thang-nhan ma giu nguyen do pho bien va so nhan.
_rs = np.random.default_rng(0)
null = np.array([_stat(_rs.permutation(_th)) for _ in range(400)])
lo, hi = np.percentile(null, [2.5, 97.5])
_m = np.nanmedian([( _Y[l][np.isin(_th,[12,1,2])].mean() / _Y[l][np.isin(_th,[6,7,8])].mean())
                   for l in G_MUA])
_d = np.nanmedian([( _Y[l][np.isin(_th,[12,1,2])].mean() / _Y[l][np.isin(_th,[6,7,8])].mean())
                   for l in G_DOI])
print('')
print(f'  ty le DONG/HE — trung vi:  theo mua {_m:.3f} | doi chung {_d:.3f}')
print(f'  quan sat duoc (mua - doi chung): {obs:+.3f}')
print(f'  dai NULL 95% (400 lan trao thang): [{lo:+.3f}, {hi:+.3f}]')
_mua_con = obs > hi
print('  ---')
print('  Nguong lay tu chinh phan phoi NULL, khong dat bang tay: nhom nho + nhan hiem thi ty le')
print('  dong/he dao dong manh, mot con so co dinh se bao dong gia.')
print(f'  KET LUAN: {"CON mua vu -> mot cot `thang` la du" if _mua_con else "DA XOA mua vu -> DONG"}')

377,110 anh co ca nhan lan index_time
phan bo thang: {1: 32529, 2: 29439, 3: 31629, 4: 31383, 5: 32230, 6: 31155, 7: 31958, 8: 31863, 9: 30620, 10: 31714, 11: 31134, 12: 31456}

nhom DANG LE theo mua (4): ['Consolidation', 'Edema', 'Pneumonia', 'Tuberculosis']
nhom DOI CHUNG        (8): ['Calcification of the Aorta', 'Cardiomegaly', 'Clavicle Fracture', 'Fracture', 'Hernia', 'Nodule', 'Rib Fracture', 'Support Devices']

  ty le DONG/HE — trung vi:  theo mua 1.004 | doi chung 1.025
  quan sat duoc (mua - doi chung): -0.021
  dai NULL 95% (400 lan trao thang): [-0.027, +0.025]
  ---
  Nguong lay tu chinh phan phoi NULL, khong dat bang tay: nhom nho + nhan hiem thi ty le
  dong/he dao dong manh, mot con so co dinh se bao dong gia.
  KET LUAN: DA XOA mua vu -> DONG


## C5 · Ghi `REVIEW_TEMPORAL.md` vào repo

In [6]:
L = []; w = L.append
w('# REVIEW · trục thời gian — bốn phép đo chặn (`nb42`)'); w('')
w('Số học dùng xuyên suốt: đòn bẩy chạm `k` nhãn phải nhấc mỗi nhãn `0.0007 × 40 / k` '
  'chỉ để vượt biên độ seed của macro.'); w('')
w('## C1 · phim chụp trước'); w('```')
w(f'study co it nhat 1 phim truoc : {B.co_truoc.mean():.1%}')
w(f'khoang cach trung vi          : {B.gap_ngay.median():.1f} ngay')
w(f'co phim truoc trong 90 ngay   : {(B.gap_ngay <= 90).mean():.1%}')
w(f'k hieu dung                   : {40*B.co_truoc.mean():.0f}')
w(f'lift/nhan can                 : {need(40*B.co_truoc.mean()):.4f}')
w('```')
w(f'**{"ĐI TIẾP" if B.co_truoc.mean() >= 0.60 else "ĐÓNG" if B.co_truoc.mean() < 0.40 else "VÙNG XÁM"}**'); w('')
w('## C2 · `lab_*_7d` có phải bản sao'); w('```')
w(f'ty le trung voi lab_*_24h (co trong so): {_tb:.1%}')
w(f'so o ma 7d them thong tin that         : {_extra:,}')
w('```')
w('Nếu ≥90% trùng thì **xu hướng không được mã hoá ở đâu cả**, và một phần của 90 cột mà `nb35` '
  'đo là đóng góp 0 chỉ đơn giản là **cột trùng lặp**.'); w('')
w('## C3 · xu hướng có tính được không'); w('```')
w(f'ty le study co >=2 lan do (xet nghiem thuan loi nhat): 24h {_p24:.1%} | 7d {_p7:.1%}')
w('```'); w('')
w('## C4 · mùa vụ')
w('```')
w(f'ty le dong/he — theo mua {_m:.3f} | doi chung {_d:.3f}')
w(f'quan sat (mua - doi chung) {obs:+.3f} | dai NULL 95% [{lo:+.3f}, {hi:+.3f}]')
w('```')
w(f'**{"CÒN mùa vụ — một cột `tháng` là đủ" if _mua_con else "ĐÃ XOÁ mùa vụ → ĐÓNG"}** '
  '(ngưỡng lấy từ phân phối null bằng cách tráo tháng, không đặt bằng tay)')
w('')
w('## Những gì phép đo này KHÔNG trả lời'); w('')
w('- **`lb672`** cần GPU để trích đặc trưng — không nằm trong notebook này.')
w('- **Cửa sổ `MICRO_FUTURE_H`** (`{0,24,48,72}`) cần truy vấn lại BigQuery: parquet hiện tại '
  'không lưu thời điểm của từng kết quả cấy. `29b` ghi đó là quét cho Stage 2, chưa ai quét.')
w('- **Chuỗi thời gian đầy đủ** (kiểu LSTM của MedFuse) cần trích lại từ `labevents`; `C3` chỉ '
  'cho **chặn trên** về việc nó có định nghĩa được không.')
w('- Phim chụp trước **có mặt** không có nghĩa là nó **giúp**. `C1` chỉ đặt chặn trên cho `k`.')
TXT = '\n'.join(L)
P_OUT = os.path.join(os.getcwd(), 'REVIEW_TEMPORAL.md')
open(P_OUT, 'w', encoding='utf-8').write(TXT)
print(TXT); print(f'\nda ghi {P_OUT}')

# REVIEW · trục thời gian — bốn phép đo chặn (`nb42`)

Số học dùng xuyên suốt: đòn bẩy chạm `k` nhãn phải nhấc mỗi nhãn `0.0007 × 40 / k` chỉ để vượt biên độ seed của macro.

## C1 · phim chụp trước
```
study co it nhat 1 phim truoc : 71.3%
khoang cach trung vi          : 3.3 ngay
co phim truoc trong 90 ngay   : 56.3%
k hieu dung                   : 29
lift/nhan can                 : 0.0010
```
**ĐI TIẾP**

## C2 · `lab_*_7d` có phải bản sao
```
ty le trung voi lab_*_24h (co trong so): 100.0%
so o ma 7d them thong tin that         : 378,389
```
Nếu ≥90% trùng thì **xu hướng không được mã hoá ở đâu cả**, và một phần của 90 cột mà `nb35` đo là đóng góp 0 chỉ đơn giản là **cột trùng lặp**.

## C3 · xu hướng có tính được không
```
ty le study co >=2 lan do (xet nghiem thuan loi nhat): 24h 18.1% | 7d 45.5%
```

## C4 · mùa vụ
```
ty le dong/he — theo mua 1.004 | doi chung 1.025
quan sat (mua - doi chung) -0.021 | dai NULL 95% [-0.027, +0.025]
```
**ĐÃ XOÁ mùa vụ → ĐÓNG** (ngưỡng lấy từ phân